# Advanced Problems with Solutions: Python Sequence Types

This notebook contains advanced practice problems on Python sequence types, iterables, mutability, immutability, indexing, slicing, concatenation, repetition, searching, ordering, and hashability.

## Best-practice format used

Each problem includes:

1. A realistic prompt.
2. Clear constraints.
3. A solution strategy.
4. A complete reference implementation or analysis.
5. Runnable checks where appropriate.

Recommended workflow:

- Try the problem first.
- Run the provided tests.
- Compare your approach with the solution.
- Modify the examples to test edge cases.

## Problem 1 — Predicting Mutation Through an Immutable Container

A tuple is immutable, but it may contain mutable objects.

Consider this object:

```python
data = ([1, 2], [3, 4], "py")
alias = data[0]
alias.append(99)
```

### Tasks

1. Predict the final value of `data`.
2. Explain why this does not violate tuple immutability.
3. Write a function `safe_tuple_copy(t)` that returns a tuple where any list elements are shallow-copied, so mutating the original list later does not affect the copied tuple.

In [1]:
# Solution to Problem 1

data = ([1, 2], [3, 4], "py")
alias = data[0]
alias.append(99)

data

([1, 2, 99], [3, 4], 'py')

### Explanation

The final value is:

```python
([1, 2, 99], [3, 4], "py")
```

The tuple still contains the same three object references. We did not replace `data[0]`; we mutated the list object that `data[0]` refers to. Tuple immutability prevents item reassignment, deletion, or insertion, but it does not recursively freeze the contained objects.

In [2]:
def safe_tuple_copy(t):
    """Return a tuple with list elements shallow-copied.

    This protects the new tuple from later mutations to top-level list elements
    in the original tuple.
    """
    return tuple(item.copy() if isinstance(item, list) else item for item in t)


original = ([1, 2], [3, 4], "py")
copied = safe_tuple_copy(original)

original[0].append(99)

print("original:", original)
print("copied:  ", copied)

assert original == ([1, 2, 99], [3, 4], "py")
assert copied == ([1, 2], [3, 4], "py")

original: ([1, 2, 99], [3, 4], 'py')
copied:   ([1, 2], [3, 4], 'py')


## Problem 2 — Sequence or Merely Iterable?

A sequence is iterable, but not every iterable is a sequence.

### Tasks

For each object below, decide whether it is:

- iterable
- a sequence
- ordered by index
- guaranteed to preserve insertion/order semantics when iterated

```python
objects = [
    [10, 20, 30],
    (10, 20, 30),
    "abc",
    range(3),
    {10, 20, 30},
    {"a": 1, "b": 2}
]
```

Then write a function `classify(obj)` that returns a dictionary with practical checks for sequence behavior.

In [3]:
from collections.abc import Iterable, Sequence

def classify(obj):
    """Classify an object using runtime protocol checks.

    Note:
    - Sequence is a stronger protocol than Iterable.
    - Dictionaries preserve insertion order in modern Python, but they are not
      sequences because they are not indexed by integer position.
    """
    return {
        "type": type(obj).__name__,
        "is_iterable": isinstance(obj, Iterable),
        "is_sequence": isinstance(obj, Sequence),
        "supports_integer_index_0": _supports_index_zero(obj),
        "has_len": hasattr(obj, "__len__")
    }


def _supports_index_zero(obj):
    try:
        obj[0]
        return True
    except Exception:
        return False


objects = [
    [10, 20, 30],
    (10, 20, 30),
    "abc",
    range(3),
    {10, 20, 30},
    {"a": 1, "b": 2}
]

for obj in objects:
    print(classify(obj))

{'type': 'list', 'is_iterable': True, 'is_sequence': True, 'supports_integer_index_0': True, 'has_len': True}
{'type': 'tuple', 'is_iterable': True, 'is_sequence': True, 'supports_integer_index_0': True, 'has_len': True}
{'type': 'str', 'is_iterable': True, 'is_sequence': True, 'supports_integer_index_0': True, 'has_len': True}
{'type': 'range', 'is_iterable': True, 'is_sequence': True, 'supports_integer_index_0': True, 'has_len': True}
{'type': 'set', 'is_iterable': True, 'is_sequence': False, 'supports_integer_index_0': False, 'has_len': True}
{'type': 'dict', 'is_iterable': True, 'is_sequence': False, 'supports_integer_index_0': False, 'has_len': True}


### Solution notes

- `list`, `tuple`, `str`, and `range` are sequence types.
- `set` is iterable but not a sequence because it has no positional indexing.
- `dict` is iterable over keys and preserves insertion order in modern Python, but it is not a sequence because `d[0]` means key lookup, not positional indexing.

## Problem 3 — Robust Searching Without Crashing

The `index` method raises `ValueError` when an element is not found.

### Task

Write `find_all(seq, target, start=0, stop=None)` that returns all positions where `target` occurs in a sequence.

Requirements:

1. It must not crash when the target is absent.
2. It must support `start` and `stop`.
3. It must work for strings, tuples, and lists.
4. It should use `index` internally when available.

In [4]:
def find_all(seq, target, start=0, stop=None):
    """Return all indexes of target in seq[start:stop].

    Uses seq.index repeatedly. Works for list, tuple, and str.
    """
    if stop is None:
        stop = len(seq)

    indexes = []
    pos = start

    while pos < stop:
        try:
            found = seq.index(target, pos, stop)
        except ValueError:
            break
        indexes.append(found)
        pos = found + 1

    return indexes


assert find_all("banana", "a") == [1, 3, 5]
assert find_all("banana", "a", 2, 5) == [3]
assert find_all([1, 2, 1, 3, 1], 1) == [0, 2, 4]
assert find_all((10, 20, 10), 99) == []

print("All tests passed.")

All tests passed.


### Why this is better than a single `index` call

A single call to `seq.index(target)` only finds the first occurrence. This solution repeatedly resumes searching one position after the previous match and safely stops when `ValueError` is raised.

## Problem 4 — Comparable vs Non-Comparable Elements

`min` and `max` require elements to be comparable, unless a `key` function is supplied.

### Tasks

1. Explain why `min([1+1j, 2+2j])` fails.
2. Write `min_by_magnitude(values)` for complex numbers.
3. Write `safe_min(values, key=None, default=None)` that behaves like `min`, but returns `default` for empty iterables and returns a readable error message for non-comparable values.

In [5]:
def min_by_magnitude(values):
    """Return the complex number with the smallest magnitude."""
    return min(values, key=abs)


assert min_by_magnitude([3+4j, 1+1j, 2+0j]) == 1+1j
print(min_by_magnitude([3+4j, 1+1j, 2+0j]))

(1+1j)


In [6]:
def safe_min(values, key=None, default=None):
    """A safer wrapper around min.

    - Returns default for an empty iterable.
    - Returns a readable string if comparison fails.
    """
    iterator = iter(values)

    try:
        first = next(iterator)
    except StopIteration:
        return default

    try:
        return min([first, *iterator], key=key)
    except TypeError as exc:
        return f"Cannot compute minimum: {exc}"


print(safe_min([], default="empty"))
print(safe_min([3, 1, 2]))
print(safe_min([1+1j, 2+2j]))
print(safe_min([1+1j, 2+2j], key=abs))

assert safe_min([], default=None) is None
assert safe_min([3, 1, 2]) == 1
assert safe_min([1+1j, 2+2j], key=abs) == 1+1j

empty
1
Cannot compute minimum: '<' not supported between instances of 'complex' and 'complex'
(1+1j)


### Solution notes

Complex numbers do not define a natural less-than ordering in Python. However, a key function such as `abs` maps each complex number to a comparable numeric magnitude.

## Problem 5 — Concatenation, Type Preservation, and Normalization

Sequence concatenation with `+` requires compatible sequence types.

### Task

Write `concat_as(target_type, *items)` that concatenates multiple sequence-like inputs after converting them to a common target type.

Examples:

```python
concat_as(tuple, [1, 2], range(3, 5), (5, 6))
# expected: (1, 2, 3, 4, 5, 6)

concat_as(str, "py", ["t", "h"], ("o", "n"))
# expected: "python"
```

Constraints:

- `target_type` may be `list`, `tuple`, or `str`.
- For `str`, every element must already be string-like after flattening.

In [7]:
def concat_as(target_type, *items):
    """Concatenate sequence-like inputs as list, tuple, or str."""
    if target_type not in {list, tuple, str}:
        raise TypeError("target_type must be list, tuple, or str")

    flattened = []

    for item in items:
        if isinstance(item, str):
            # Strings are sequences, but for str output we want characters;
            # for list/tuple output, list('py') is also reasonable here.
            flattened.extend(item)
        else:
            flattened.extend(item)

    if target_type is list:
        return flattened

    if target_type is tuple:
        return tuple(flattened)

    if target_type is str:
        if not all(isinstance(x, str) for x in flattened):
            raise TypeError("all elements must be strings to concatenate as str")
        return "".join(flattened)


assert concat_as(tuple, [1, 2], range(3, 5), (5, 6)) == (1, 2, 3, 4, 5, 6)
assert concat_as(list, (1, 2), range(3)) == [1, 2, 0, 1, 2]
assert concat_as(str, "py", ["t", "h"], ("o", "n")) == "python"

print("All tests passed.")

All tests passed.


### Design discussion

This function makes the conversion explicit. That is usually better than relying on `+` when inputs may be heterogeneous, because Python intentionally avoids guessing whether a tuple plus a list should produce a tuple or a list.

## Problem 6 — The Repetition Aliasing Trap

List repetition copies references, not the objects themselves.

### Task

Create a function `make_matrix(rows, cols, fill=0)` that returns a matrix whose rows are independent lists.

Then demonstrate why this implementation is wrong:

```python
bad = [[0] * cols] * rows
```

In [8]:
def make_matrix(rows, cols, fill=0):
    """Create a matrix with independent rows."""
    return [[fill for _ in range(cols)] for _ in range(rows)]


good = make_matrix(3, 4)
good[0][0] = 99

print(good)

assert good == [
    [99, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0]
]

[[99, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]


In [9]:
# Demonstrating the wrong approach

rows, cols = 3, 4
bad = [[0] * cols] * rows

bad[0][0] = 99

print(bad)
print("All rows are the same object:", bad[0] is bad[1] is bad[2])

assert bad == [
    [99, 0, 0, 0],
    [99, 0, 0, 0],
    [99, 0, 0, 0]
]

[[99, 0, 0, 0], [99, 0, 0, 0], [99, 0, 0, 0]]
All rows are the same object: True


### Explanation

`[[0] * cols] * rows` repeats the same inner list reference. Mutating one row mutates the shared row object visible at every repeated position.

## Problem 7 — Advanced Slicing: Implement Every-Nth Window

Slicing supports `start`, `stop`, and `step`.

### Task

Write `windowed_stride(seq, size, step=1)` that returns overlapping windows from a sequence.

Examples:

```python
windowed_stride("abcdef", 3)
# ["abc", "bcd", "cde", "def"]

windowed_stride("abcdef", 3, 2)
# ["abc", "cde"]
```

Requirements:

- Preserve the slice type where Python slicing naturally does so.
- Raise `ValueError` if `size <= 0` or `step <= 0`.

In [10]:
def windowed_stride(seq, size, step=1):
    """Return overlapping windows of length size, advancing by step."""
    if size <= 0:
        raise ValueError("size must be positive")
    if step <= 0:
        raise ValueError("step must be positive")

    return [seq[i:i + size] for i in range(0, len(seq) - size + 1, step)]


assert windowed_stride("abcdef", 3) == ["abc", "bcd", "cde", "def"]
assert windowed_stride("abcdef", 3, 2) == ["abc", "cde"]
assert windowed_stride([1, 2, 3, 4], 2) == [[1, 2], [2, 3], [3, 4]]
assert windowed_stride((1, 2, 3, 4), 2) == [(1, 2), (2, 3), (3, 4)]
assert windowed_stride(range(6), 3, 2) == [range(0, 3), range(2, 5)]

print("All tests passed.")

All tests passed.


### Solution notes

Because the function uses native slicing, each window naturally has the same slice behavior as the original object:

- string slices return strings
- list slices return lists
- tuple slices return tuples
- range slices return ranges

## Problem 8 — Making Nested Data Hashable

A tuple is hashable only if all its elements are hashable.

### Task

Write `deep_freeze(obj)` that converts common mutable containers into immutable, hashable equivalents:

- `list` → `tuple`
- `set` → `frozenset`
- `dict` → sorted tuple of key-value pairs
- recursively freeze nested values

Then use it to place nested data into a set.

In [11]:
def deep_freeze(obj):
    """Recursively convert common mutable containers into hashable structures."""
    if isinstance(obj, list):
        return tuple(deep_freeze(item) for item in obj)

    if isinstance(obj, tuple):
        return tuple(deep_freeze(item) for item in obj)

    if isinstance(obj, set):
        return frozenset(deep_freeze(item) for item in obj)

    if isinstance(obj, dict):
        return tuple(
            sorted(
                (deep_freeze(key), deep_freeze(value))
                for key, value in obj.items()
            )
        )

    return obj


nested = {
    "language": "Python",
    "topics": ["sequence", "iterable", "hashing"],
    "flags": {"advanced", "practice"},
    "meta": {"version": 1}
}

frozen = deep_freeze(nested)

print(frozen)
print(hash(frozen))

seen = {frozen}
assert frozen in seen

(('flags', frozenset({'advanced', 'practice'})), ('language', 'Python'), ('meta', (('version', 1),)), ('topics', ('sequence', 'iterable', 'hashing')))
-7330819254682114668


### Design discussion

Dictionaries are converted to sorted tuples of key-value pairs so that logically equivalent dictionaries produce the same frozen representation regardless of insertion order.

This works only when the frozen keys are sortable with each other. A production-grade version may sort using `repr(key)` or preserve insertion order depending on the desired semantics.

## Problem 9 — Membership Testing: Linear Search vs Range Arithmetic

Most sequence membership checks scan items, but `range` can test membership arithmetically.

### Task

Implement `in_arithmetic_range(x, start, stop, step=1)` without constructing a list or using `range`.

It should match Python's `x in range(start, stop, step)` behavior for integer inputs.

In [12]:
def in_arithmetic_range(x, start, stop, step=1):
    """Return True if x belongs to range(start, stop, step)."""
    if step == 0:
        raise ValueError("step cannot be zero")

    if step > 0:
        if x < start or x >= stop:
            return False
    else:
        if x > start or x <= stop:
            return False

    return (x - start) % step == 0


test_cases = [
    (4, 0, 10, 2),
    (5, 0, 10, 2),
    (10, 0, 10, 1),
    (3, 10, 0, -1),
    (0, 10, 0, -1),
    (10, 10, 0, -1)
]

for x, start, stop, step in test_cases:
    expected = x in range(start, stop, step)
    actual = in_arithmetic_range(x, start, stop, step)
    print((x, start, stop, step), actual, expected)
    assert actual == expected

print("All tests passed.")

(4, 0, 10, 2) True True
(5, 0, 10, 2) False False
(10, 0, 10, 1) False False
(3, 10, 0, -1) True True
(0, 10, 0, -1) False False
(10, 10, 0, -1) True True
All tests passed.


### Solution notes

For `x` to be in the range:

1. It must lie within the half-open interval defined by `start`, `stop`, and the sign of `step`.
2. The distance from `start` must be divisible by `step`.

## Problem 10 — Build a Minimal Custom Sequence

A custom class can behave like a sequence by implementing the right special methods.

### Task

Create a class `CyclicSequence` that wraps a finite sequence but supports circular indexing:

```python
c = CyclicSequence(["a", "b", "c"])
c[0]   # "a"
c[3]   # "a"
c[4]   # "b"
c[-1]  # "c"
```

Requirements:

- Support `len(c)`.
- Support iteration.
- Support integer indexing.
- Support slicing by returning a list.
- Reject empty input.

In [13]:
class CyclicSequence:
    """A read-only sequence-like wrapper with circular integer indexing."""

    def __init__(self, values):
        self._values = tuple(values)
        if len(self._values) == 0:
            raise ValueError("CyclicSequence cannot be empty")

    def __len__(self):
        return len(self._values)

    def __iter__(self):
        return iter(self._values)

    def __getitem__(self, index):
        if isinstance(index, slice):
            # For slices, use normal slicing over the finite backing sequence.
            return list(self._values[index])

        if not isinstance(index, int):
            raise TypeError("index must be an integer or slice")

        return self._values[index % len(self._values)]

    def __repr__(self):
        return f"CyclicSequence({list(self._values)!r})"


c = CyclicSequence(["a", "b", "c"])

assert len(c) == 3
assert list(c) == ["a", "b", "c"]
assert c[0] == "a"
assert c[3] == "a"
assert c[4] == "b"
assert c[-1] == "c"
assert c[:2] == ["a", "b"]

print(c)
print(c[0], c[3], c[4], c[-1], c[:2])

CyclicSequence(['a', 'b', 'c'])
a a b c ['a', 'b']


### Extension challenge

Modify `CyclicSequence` so that a slice such as `c[2:8]` returns circular values:

```python
["c", "a", "b", "c", "a", "b"]
```

This requires interpreting the slice bounds manually instead of delegating directly to the backing tuple.

## Final Review Checklist

You should now be able to:

- Distinguish iterable objects from sequence objects.
- Explain why tuple immutability is shallow.
- Use `index` safely.
- Predict behavior of slicing, concatenation, and repetition.
- Avoid aliasing bugs caused by repeated mutable objects.
- Explain when `min`, `max`, and `hash` fail.
- Convert nested mutable structures into hashable representations.
- Implement a minimal custom sequence-like class.